# 07 — SHAP (Explicabilidade)

Como KMeans é não supervisionado, treinamos um modelo supervisionado para prever `cluster_id` usando as mesmas features pré-processadas.
Depois, usamos SHAP para entender quais variáveis levam a cada cluster.


In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve()))

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

import shap

from src.features import build_modeling_dataframe
from src.modeling import get_feature_names, load_pipeline, transform_for_model

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
SHAP_DIR = REPORTS_DIR / 'shap'
SHAP_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_parquet(PROCESSED_DIR / 'base_cliente_clusterizada.parquet')
pipe = load_pipeline(str(REPORTS_DIR / 'kmeans_pipeline.joblib'))
X_raw, _ = build_modeling_dataframe(df)
X = transform_for_model(pipe, X_raw)
y = df['cluster_id'].astype(int).values

feature_names = get_feature_names(pipe)
X.shape, len(feature_names)


## 1) Modelo supervisionado para prever cluster

Usamos regressão logística multinomial (saga) por compatibilidade com matrizes esparsas do OneHotEncoder.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(
    max_iter=2000,
    solver='saga',
    n_jobs=-1,
)
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
acc = accuracy_score(y_test, pred)
cm = confusion_matrix(y_test, pred)
acc, cm


## 2) SHAP

Calculamos SHAP em uma amostra para manter o custo computacional sob controle.


In [ ]:
sample_n = min(3000, X_train.shape[0])
rng = np.random.default_rng(42)
idx = rng.choice(X_train.shape[0], size=sample_n, replace=False)
X_shap = X_train[idx]

explainer = shap.Explainer(clf, X_train, feature_names=feature_names)
shap_values = explainer(X_shap)
shap_values


In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig(SHAP_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()


## 3) Salvando métricas de importância (mean |SHAP|)

Armazenamos um resumo agregando por feature e classe.


In [ ]:
vals = shap_values.values
if vals.ndim == 3:
    out = []
    for c in range(vals.shape[2]):
        imp = np.mean(np.abs(vals[:, :, c]), axis=0)
        out.append(pd.DataFrame({'feature': feature_names, 'mean_abs_shap': imp, 'classe': c}))
    imp_df = pd.concat(out, ignore_index=True).sort_values(['classe','mean_abs_shap'], ascending=[True, False])
else:
    imp = np.mean(np.abs(vals), axis=0)
    imp_df = pd.DataFrame({'feature': feature_names, 'mean_abs_shap': imp}).sort_values('mean_abs_shap', ascending=False)

imp_df.to_parquet(SHAP_DIR / 'shap_importancia.parquet', index=False)
imp_df.head(20)
